# exp/r2-dmix on Colab (T4): queue [gru_v8p2z3c, gru_v8p2z3d]
Runtime -> Change runtime type -> T4 GPU, then Run all. A failed run raises with the config name and stops the queue (no silent skip). After a disconnect, just Run all again: each config resumes from its `last.pt` on Drive (`--resume`; model, optimizer, RNG), and a config whose `train_log.csv` already reached the final epoch or has an `early_stop` row is skipped. Each run writes `best.pt` (best validation PESQ, inference weights only), `last.pt` (full state, every epoch, ~28 MB) and `train_log.csv`; console output is appended to `train_stdout.log` there.

Queue (2026-11-24, third batch), both continuing the submission gru_v8p2z3b (val 3.6063 / val_remix 3.4628 with rms_norm; train64 4.01, gap 0.40; still rising at the 200-epoch cap, best epoch 179): init from weights/gru_v8p2z3b.pt (weights only, fresh Adam lr 5e-5), everything else as gru_v8p2z3b (0.2 cmag + 0.32 torch-pesq, full dynamic mixing, train-time rms_norm 0.079130), seed 0, 200 epochs, patience 30. Clean-side augmentation is applied to the clean utterance before mixing; the target is the augmented clean.
- **gru_v8p2z3c**: same augmentation as gru_v8p2z3b (continuation): speed perturbation p 0.5, factor U[0.9, 1.1]; random EQ p 0.5 (low-shelf U[150, 500] Hz, high-shelf U[2, 5] kHz, gains U[-6, +6] dB, RMS kept).
- **gru_v8p2z3d**: stronger clean-side augmentation: speed perturbation p 0.7, factor U[0.85, 1.15] (61 factors, 60 cached anti-aliasing FIRs, ~11 MB); random EQ p 0.7, gains U[-9, +9] dB (same corners).

Estimated T4 time: gru_v8p2z3a/b took a median 19.4 s training + 18.9 s validation/train64 per epoch on Colab (~38 s), so **about 2.1 h (z3c) and about 2.2 h (z3d, slightly more resampling/EQ work on the CPU)** for 200 epochs, less with an early stop; about 4.3 h for the queue plus about 5 min setup. If the session limit cuts it, Run all again (resume).

After training: copy `nrc_ckpt/<name>/best.pt` to `weights/<name>.pt` (the configs point there) and `train_log.csv` to `logs/train_log_v8p2z3{c,d}.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
import os
!rm -rf /content/repo && git clone -b exp/r2-dmix --depth 1 https://github.com/pauljh4323/SGSYS_13thTeam_NoiseReductionChallenge.git /content/repo
%cd /content/repo
for w in ('weights/gru_v8p2z3b.pt',):  # init_from of gru_v8p2z3c / gru_v8p2z3d
    assert os.path.isfile(w) and os.path.getsize(w) > 1_000_000, f'{w} missing (init_from)'
    print('init weights OK:', w, os.path.getsize(w), 'bytes')
print('CPUs:', os.cpu_count())

In [ ]:
!pip install -q pesq==0.0.4 pystoi soundfile pyyaml
# torch-pesq without its torchaudio dependency (src/nrc/tpesq.py provides the pieces it uses); see requirements-dpesq.txt
!pip install -q --no-deps -r requirements-dpesq.txt
import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, "src"); from nrc.tpesq import pesq_loss; import torch; '
                    'l = pesq_loss(1.0, 16000); x = torch.randn(2, 48000) * 0.1; print("torch-pesq OK, MOS(x, x) =", l.mos(x, x).tolist())'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, 'torch-pesq import / smoke failed'

In [ ]:
!mkdir -p /content/data && cp -r --update=none /content/drive/MyDrive/noise_reduction_challenge /content/data/

In [ ]:
import os, yaml, pandas as pd
from pathlib import Path
CONFIGS = ["configs/dl/gru_v8p2z3c.yaml", "configs/dl/gru_v8p2z3d.yaml"]  # run in this order
for cfg in CONFIGS:
    assert os.path.isfile(cfg), f"{cfg} missing (wrong branch?)"
DATA = '/content/data/noise_reduction_challenge'
CKPT_ROOT = Path('/content/drive/MyDrive/nrc_ckpt')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
for cfg in CONFIGS:
    lp = CKPT_ROOT / Path(cfg).stem / 'last.pt'
    print(Path(cfg).stem, 'resume point:', lp, 'found' if lp.is_file() else 'not found (fresh start from init_from)')
import subprocess, sys

def run_logged(cmd, log, name):
    """Run cmd, stream its output here and append it to log; raise if it exits non-zero (no silent `| tee`)."""
    print('$ ' + ' '.join(map(str, cmd)), flush=True)
    with open(log, 'a', encoding='utf-8') as f:
        proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            f.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{name} failed with exit code {rc}; see {log}')

In [ ]:
def ckpt_dir(cfg): return CKPT_ROOT / Path(cfg).stem

def finished(cfg):
    log = ckpt_dir(cfg) / 'train_log.csv'
    epochs = yaml.safe_load(open(cfg))['train']['epochs']
    if not log.is_file() or os.path.getsize(log) == 0:
        return False
    df = pd.read_csv(log)
    if len(df) == 0:
        return False
    early = 'early_stop' in df and (pd.to_numeric(df.early_stop, errors='coerce') == 1).any()
    return bool(early) or df.epoch.max() >= epochs

for cfg in CONFIGS:
    d = ckpt_dir(cfg)
    if finished(cfg):
        print(f'skip {cfg}: final epoch reached or early-stopped')
        continue
    d.mkdir(parents=True, exist_ok=True)
    run_logged([sys.executable, 'scripts/train_dpesq.py', '--config', cfg, '--data-root', DATA, '--ckpt-dir', d,
                '--resume', '--device', 'auto'], d / 'train_stdout.log', Path(cfg).stem)

In [ ]:
for cfg in CONFIGS:
    log = ckpt_dir(cfg) / 'train_log.csv'
    if not log.is_file():
        print(f'{Path(cfg).stem}: no train_log.csv'); continue
    df = pd.read_csv(log).dropna(subset=['val_pesq'])
    if df.empty:
        print(f'{Path(cfg).stem}: no validation yet'); continue
    r = df.loc[df.val_pesq.idxmax()]
    print(f'{Path(cfg).stem}: best epoch {int(r.epoch)}, val PESQ {r.val_pesq:.4f}, train64 PESQ {r.train64_pesq:.4f} '
          f'(gap {r.gap:+.4f}), val SNR {r.val_snr:.2f} dB, tpesq MOS {r.tpesq_mos:.3f} '
          f'(last epoch {int(pd.read_csv(log).epoch.max())})')